# 02 - Data Cleaning
Neeche wahi 8 steps hain jo `src/clean_data.py` me hain - audit ke saath.

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv('../data/raw/event_log.csv')
before = len(df); print('rows before:', before)

In [ ]:
df.columns = [c.strip().lower() for c in df.columns]
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
for c in ['wait_minutes','service_minutes','cost_inr']: df[c] = pd.to_numeric(df[c], errors='coerce')

In [ ]:
n=len(df); df = df.dropna(subset=['case_id','activity','timestamp']); print('missing key rows dropped:', n-len(df))
n=len(df); df = df.drop_duplicates(['case_id','activity','timestamp']); print('duplicates removed:', n-len(df))
df['resource'] = df['resource'].fillna('UNKNOWN').str.upper()
df['department'] = df['department'].fillna('Unassigned')

In [ ]:
df.loc[df.wait_minutes<0,'wait_minutes']=0
df['service_minutes'] = df['service_minutes'].where(df.service_minutes>0, np.nan).fillna(df.service_minutes.median())
df['cost_inr'] = df['cost_inr'].clip(lower=0)

In [ ]:
df = df.sort_values(['case_id','timestamp'])
df['event_index'] = df.groupby('case_id').cumcount()
df['is_rework'] = (df.groupby('case_id')['activity'].shift(1)==df['activity']).astype(int)
print('rows after:', len(df), '| rework events:', df.is_rework.sum())
df.to_csv('../data/processed/event_log_clean.csv', index=False)